# 🔍 RAG Básico con ChromaDB persistente

In [15]:
import os
import warnings

import requests
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama

from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

warnings.filterwarnings("ignore")

# ── Rutas ──────────────────────────────────────────────────────────────────
BASE_DIR    = "/Users/guane/Documentos/GlogalLogic/AI-course"
PDF_DIR     = os.path.join(BASE_DIR, "data", "Papers")
VECTOR_DIR  = os.path.join(BASE_DIR, "data", "vector_db")   # 💾 BD persistente

os.makedirs(PDF_DIR,    exist_ok=True)
os.makedirs(VECTOR_DIR, exist_ok=True)

print(f"📁 PDFs    → {PDF_DIR}")
print(f"📁 Chroma  → {VECTOR_DIR}")

📁 PDFs    → /Users/guane/Documentos/GlogalLogic/AI-course/data/Papers
📁 Chroma  → /Users/guane/Documentos/GlogalLogic/AI-course/data/vector_db


In [16]:
urls = [
    "https://arxiv.org/pdf/2306.06031v1.pdf",  # FinGPT
    "https://arxiv.org/pdf/2306.12156v1.pdf",
    "https://arxiv.org/pdf/2306.14289v1.pdf",
    "https://arxiv.org/pdf/2305.10973v1.pdf",
    "https://arxiv.org/pdf/2306.13643v1.pdf",
]

ml_papers = []

for i, url in enumerate(urls):
    filename = os.path.join(PDF_DIR, f"paper{i+1}.pdf")

    if not os.path.exists(filename):
        print(f"⬇️  Descargando {filename} ...")
        response = requests.get(url, timeout=60)
        with open(filename, "wb") as f:
            f.write(response.content)
    else:
        print(f"✅ Ya existe: {filename}")

    loader = PyPDFLoader(filename)
    ml_papers.extend(loader.load())

print(f"\n📄 Total páginas cargadas: {len(ml_papers)}")

✅ Ya existe: /Users/guane/Documentos/GlogalLogic/AI-course/data/Papers/paper1.pdf
✅ Ya existe: /Users/guane/Documentos/GlogalLogic/AI-course/data/Papers/paper2.pdf
✅ Ya existe: /Users/guane/Documentos/GlogalLogic/AI-course/data/Papers/paper3.pdf
✅ Ya existe: /Users/guane/Documentos/GlogalLogic/AI-course/data/Papers/paper4.pdf
✅ Ya existe: /Users/guane/Documentos/GlogalLogic/AI-course/data/Papers/paper5.pdf

📄 Total páginas cargadas: 57


In [17]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
    length_function=len,
)

documents = text_splitter.split_documents(ml_papers)

print(f"✂️  Total chunks generados: {len(documents)}")
print(f"\n🔍 Ejemplo (chunk #10):")
print(documents[10].page_content[:500])

✂️  Total chunks generados: 211

🔍 Ejemplo (chunk #10):
highly volatile, changing rapidly in response to news events
or market movements.
Trends, often observable through websites like Seeking
Alpha, Google Trends, and other finance-oriented blogs and
forums, offer critical insights into market movements and in-
vestment strategies. They feature:
• Analyst perspectives: These platforms provide access to
market predictions and investment advice from seasoned
financial analysts and experts.
• Market sentiment: The discourse on these platforms can
refle


In [ ]:
embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")

# ── Crea la BD vectorial y la guarda en VECTOR_DIR ──────────────────────────
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory=VECTOR_DIR,   # 💾 persiste en disco
    collection_name="ml_papers",
)

print(f"✅ Vector store creado y guardado en: {VECTOR_DIR}")
print(f"   Documentos indexados: {vectorstore._collection.count()}")

✅ Vector store creado y guardado en: /Users/guane/Documentos/GlogalLogic/AI-course/data/vector_db
   Documentos indexados: 633


### Cargar la BD ya existente (sin re-indexar)

Si ya ejecutaste la celda anterior en una sesión previa, puedes cargar la BD directamente:

In [19]:
# ── Opcional: carga desde disco sin re-crear ────────────────────────────────
# embeddings = OllamaEmbeddings(model="nomic-embed-text")
# vectorstore = Chroma(
#     persist_directory=VECTOR_DIR,
#     embedding_function=embeddings,
#     collection_name="ml_papers",
# )
# print(f"✅ Vector store cargado desde disco: {vectorstore._collection.count()} docs")

## Retriever

In [20]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)

print(f"🔎 Retriever listo | tipo: {type(retriever).__name__}")

🔎 Retriever listo | tipo: VectorStoreRetriever


## LLM + Prompt

In [21]:
llm = ChatOllama(model="llama3.2:1b", temperature=0)

prompt_template = PromptTemplate.from_template("""
Answer the question using the provided context.

Context:
{context}

Question:
{question}
""")

print("🤖 LLM y prompt listos.")

🤖 LLM y prompt listos.


## Cadena RAG (LCEL)

**Flujo:**
```
invoke({"question": "..."})  →  RunnableParallel
  ├─ context:  extrae question → retriever → format_docs
  └─ question: extrae question → pasa directo
       → prompt_template → llm → StrOutputParser
```

In [22]:
def format_docs(docs):
    """Concatena el contenido de los documentos recuperados."""
    return "\n\n".join([d.page_content for d in docs])


chain = (
    {
        # ✅ Extraemos el string antes de pasarlo al retriever
        "context":  RunnableLambda(lambda x: x["question"]) | retriever | RunnableLambda(format_docs),
        "question": RunnableLambda(lambda x: x["question"]),
    }
    | prompt_template
    | llm
    | StrOutputParser()
)

print("⛓️  Cadena RAG construida correctamente.")

⛓️  Cadena RAG construida correctamente.


## Consulta

In [23]:
question = "¿Cuál es el proceso de ingestión de datos en FinGPT?"

respuesta = chain.invoke({"question": question})

print(f"❓ Pregunta:\n{question}")
print(f"\n💬 Respuesta:\n{respuesta}")

❓ Pregunta:
¿Cuál es el proceso de ingestión de datos en FinGPT?

💬 Respuesta:
El proceso de ingestión de datos en FinGPT implica varios pasos para capturar y procesar información financiera de manera efectiva. A continuación, se describen los diferentes componentes del pipeline:

1. **Data Source Layer**: Orígenes de datos financieros variados, como sitios web de noticias, plataformas de redes sociales, estados financieros, tendencias del mercado y más.
2. **Data Engineering Layer**: Real-time procesamiento de datos NLP para manejar la alta sensibilidad temporal y baja relación de señales a nojas en el análisis financiero.
3. **LLM (Language Model)**: Componente central que incorpora técnicas de fine-tuning con un enfoque en la adaptación ligera para mantener el modelo actualizado y relevante.
4. **Application Layer**: Demostración práctica del uso de FinGPT en tareas financieras, como asesoramiento automático, trading de criptomonedas y desarrollo de aplicaciones financieras a través

## Prueba con varias preguntas

In [24]:
preguntas = [
    "¿Qué modelos de lenguaje se utilizan en FinGPT?",
    "¿Cuáles son las principales métricas de evaluación usadas?",
    "¿Qué técnicas de fine-tuning se mencionan en los papers?",
]

for q in preguntas:
    print(f"\n{'='*60}")
    print(f"❓ {q}")
    print("-"*60)
    respuesta = chain.invoke({"question": q})
    print(respuesta)


❓ ¿Qué modelos de lenguaje se utilizan en FinGPT?
------------------------------------------------------------
No se proporciona información sobre los modelos de lenguaje específicos utilizados en FinGPT. Sin embargo, se menciona que el modelo utiliza un enfoque "data-centric" para proporcionar acceso a datos financieros y recursos para desarrollar modelos de lenguaje fincales.

❓ ¿Cuáles son las principales métricas de evaluación usadas?
------------------------------------------------------------
Las principales métricas de evaluación utilizadas en el artículo "Drag Your GAN: Interactive Point-based Manipulation on the Generative Image Manifold SIGGRAPH ’23 Conference Proceedings, August 6–10, 2023, Los Angeles, CA, USA" son:

1. **MSE (Máximo Error):** Es una métrica de error que se utiliza para evaluar la precisión de las imágenes generadas por el GAN.
2. **LPIPS (Luminance-Perceptual Index of Visual Simplicity):** Es una métrica que evalúa la simplicidad visual de las imágenes ge